# ResNet-UNet Model Training

Training notebook for image forgery detection using ResNet encoder + UNet decoder architecture.

## Import Libraries and Shared Utilities

In [1]:
import sys
import os
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path to import utils
parent_dir = str(Path.cwd().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(f"Utils path: {parent_dir}")
print(f"Utils file exists: {os.path.exists(os.path.join(parent_dir, 'utils.py'))}\n")

# Import shared utility functions
try:
    from utils import (
        load_image_dataset,
        normalize,
        augment_image,
        load_dataset_with_info,
        BASE_PATH
    )
    print(f"✅ Successfully imported from utils.py")
except ImportError as e:
    print(f"❌ Import Error: {e}")
    print(f"   Current directory: {Path.cwd()}")
    print(f"   Parent directory: {parent_dir}")
    print(f"   Python path: {sys.path[:3]}...")
    raise


Utils path: c:\Users\Paudel\Desktop\techsprint_xhack\ml-app
Utils file exists: True

✅ Successfully imported from utils.py


## Load and Prepare Datasets

Using shared `load_dataset_with_info` function from utils.py

In [3]:
# Configuration
BATCH_SIZE = 32
IMAGE_SIZE = 224

print(f"Dataset Base Path: {BASE_PATH}")
print(f"Path exists: {BASE_PATH.exists()}\n")

# Load datasets with detailed logging
print("="*60)
print("LOADING DATASETS FOR TRAINING")
print("="*60)

print("\n📂 Loading Training Set:")
train_au_ds = load_dataset_with_info('train', 'Au', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
train_tp_ds = load_dataset_with_info('train', 'Tp', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
train_mask_ds = load_dataset_with_info('train', 'Mask', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)

print("\n📂 Loading Test Set:")
test_au_ds = load_dataset_with_info('test', 'Au', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
test_tp_ds = load_dataset_with_info('test', 'Tp', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
test_mask_ds = load_dataset_with_info('test', 'Mask', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)

print("\n📂 Loading Validation Set:")
val_au_ds = load_dataset_with_info('val', 'Au', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
val_tp_ds = load_dataset_with_info('val', 'Tp', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)
val_mask_ds = load_dataset_with_info('val', 'Mask', BASE_PATH, BATCH_SIZE, IMAGE_SIZE)

# Verify all datasets loaded
all_datasets = [
    train_au_ds, train_tp_ds, train_mask_ds,
    test_au_ds, test_tp_ds, test_mask_ds,
    val_au_ds, val_tp_ds, val_mask_ds
]

if all(ds is not None for ds in all_datasets):
    print("\n" + "="*60)
    print("✅ ALL DATASETS LOADED SUCCESSFULLY!")
    print("="*60)
else:
    print("\n⚠️  WARNING: Some datasets failed to load!")

Dataset Base Path: c:\Users\Paudel\Desktop\techsprint_xhack\ml-app\split_dataset
Path exists: True

LOADING DATASETS FOR TRAINING

📂 Loading Training Set:
Found 1181 files.
✅ train        | Au   | 1181 images | Au
Found 1238 files.
✅ train        | Tp   | 3151 images | Tp
Found 3151 files.
✅ train        | Mask | 3151 images | Mask

📂 Loading Test Set:
Found 337 files.
✅ test         | Au   |  337 images | Au
Found 324 files.
✅ test         | Tp   |  839 images | Tp
Found 839 files.
✅ test         | Mask |  839 images | Mask

📂 Loading Validation Set:
Found 170 files.
✅ val          | Au   |  170 images | Au
Found 148 files.
✅ val          | Tp   |  445 images | Tp
Found 445 files.
✅ val          | Mask |  445 images | Mask

✅ ALL DATASETS LOADED SUCCESSFULLY!


## Normalization of images

In [4]:
# Normalize all datasets
print("NORMALIZING DATASETS")

# Apply normalization to training datasets
train_au_ds = train_au_ds.map(lambda x: normalize(x), num_parallel_calls=tf.data.AUTOTUNE)
train_tp_ds = train_tp_ds.map(lambda x: normalize(x), num_parallel_calls=tf.data.AUTOTUNE)
train_mask_ds = train_mask_ds.map(lambda x: normalize(x), num_parallel_calls=tf.data.AUTOTUNE)

# Apply normalization to test datasets
test_au_ds = test_au_ds.map(lambda x: normalize(x), num_parallel_calls=tf.data.AUTOTUNE)
test_tp_ds = test_tp_ds.map(lambda x: normalize(x), num_parallel_calls=tf.data.AUTOTUNE)
test_mask_ds = test_mask_ds.map(lambda x: normalize(x), num_parallel_calls=tf.data.AUTOTUNE)

# Apply normalization to validation datasets
val_au_ds = val_au_ds.map(lambda x: normalize(x), num_parallel_calls=tf.data.AUTOTUNE)
val_tp_ds = val_tp_ds.map(lambda x: normalize(x), num_parallel_calls=tf.data.AUTOTUNE)
val_mask_ds = val_mask_ds.map(lambda x: normalize(x), num_parallel_calls=tf.data.AUTOTUNE)

# Prefetch for better performance
train_au_ds = train_au_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
train_tp_ds = train_tp_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
train_mask_ds = train_mask_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

test_au_ds = test_au_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
test_tp_ds = test_tp_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
test_mask_ds = test_mask_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

val_au_ds = val_au_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_tp_ds = val_tp_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_mask_ds = val_mask_ds.prefetch(buffer_size=tf.data.AUTOTUNE)


NORMALIZING DATASETS


## Data Argumentation

In [5]:
# Data Augmentation (applied only to training set)
print("\n" + "="*60)
print("APPLYING DATA AUGMENTATION TO TRAINING SET")
print("="*60)

# Augmentation wrapper for single images
def augment_single_image(image):
    # Use augment_image from utils by creating dummy mask
    aug_img, _ = augment_image(image, image)
    return aug_img

# Apply augmentation to training datasets (Au and Tp only, no masks needed)
train_au_ds = train_au_ds.map(augment_single_image, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
train_tp_ds = train_tp_ds.map(augment_single_image, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
print('Applied')



APPLYING DATA AUGMENTATION TO TRAINING SET
Applied
